In [ ]:
from langchain_classic.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableMap

In [ ]:
docs = TextLoader("/Users/pawanpahune/RAG_From_Scratch/Query_Enhancement_Techniques/Langchain_crewai.txt").load()
splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 50)
chunks = splitter.split_documents(docs)
chunks

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)

retriever = vectorstore.as_retriever(search_type = "mmr", search_kwargs = {"k":5})

retriever


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

llm = init_chat_model('groq:openai/gpt-oss-120b', temperature=0.7)



In [ ]:
query_expansion_prompt = PromptTemplate.from_template(
    """You are a helpful assistant. Your task is to expand the user's query to include more relevant keywords and phrases that can help retrieve more relevant documents.

    Original Query: "{query}"

    Expanded Query: """
)

query_expansion_chain = query_expansion_prompt|llm|StrOutputParser()

In [ ]:
query_expansion_chain.invoke({"query": "Langchain_Memory"})

In [ ]:
question_prompt = PromptTemplate.from_template(
    """
    Answer the question based on the retrieved context.
    Context: {context}

    Question: {input}
    """
)

document_chain = create_stuff_documents_chain(llm = llm, prompt = question_prompt)

In [ ]:
document_chain

In [ ]:
rag_pipeline = (
    RunnableMap(
        {
            "input": lambda x: x["input"],
            "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))
        }
    )
    | document_chain
)

In [ ]:
rag_pipeline

In [ ]:
query = {"input": "crew_ai agents"}
print(query_expansion_chain.invoke({"query": query}))
response = rag_pipeline.invoke(query)
print("Answer is following :",response)